# ..

In [10]:
!pip -q install pymupdf pandas

In [11]:
from google.colab import files as colab_files

uploaded = colab_files.upload()
pdf_path = next(iter(uploaded.keys()))

print("업로드된 파일:", pdf_path)

Saving 2110 사규관리규정(제17차 개정 20201120).pdf to 2110 사규관리규정(제17차 개정 20201120) (1).pdf
업로드된 파일: 2110 사규관리규정(제17차 개정 20201120) (1).pdf


In [12]:
import fitz
import re
import json
import pandas as pd
from pathlib import Path


# =========================
# 설정값
# =========================

START_PAGE = 1        # 표지/목차를 제외하고 싶으면 시작 페이지 수정
END_PAGE = None       # None이면 마지막 페이지까지
INCLUDE_SUPPLEMENTARY = False  # 부칙까지 별도 chunk로 넣고 싶으면 True


# =========================
# 정규식 패턴
# =========================

CHAPTER_RE = re.compile(r"^\s*제\s*(\d+)\s*장(?:\s+|$)(.*)$")
SECTION_RE = re.compile(r"^\s*제\s*(\d+)\s*절(?:\s+|$)(.*)$")

# 제1조(목적), 제1조 (목적), 제1조의2(정의) 대응
ARTICLE_RE = re.compile(
    r"^\s*제\s*(\d+)\s*조(?:\s*의\s*(\d+))?\s*(?:\(([^)\n]{0,80})\))?\s*(.*)$"
)

SUPPLEMENTARY_RE = re.compile(r"^\s*부\s*칙\b.*$")


# =========================
# 전처리 함수
# =========================

def normalize_page_text(text: str) -> str:
    """
    PDF에서 추출된 텍스트를 규정집 파싱에 맞게 정리한다.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ").replace("\ufeff", "")

    # PDF 추출 시 조문 제목 앞에 줄바꿈이 안 들어가는 경우 보정
    text = re.sub(
        r"(?<!\n)(제\s*\d+\s*조(?:\s*의\s*\d+)?\s*\()",
        r"\n\1",
        text
    )

    # 장/절 제목도 줄바꿈 보정
    text = re.sub(r"(?<!\n)(제\s*\d+\s*장\s+)", r"\n\1", text)
    text = re.sub(r"(?<!\n)(제\s*\d+\s*절\s+)", r"\n\1", text)

    # 공백 정리
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def is_noise_line(line: str) -> bool:
    """
    목차, 페이지 번호, 점선 등 색인에 불필요한 줄 제거.
    """
    stripped = line.strip()

    if not stripped:
        return True

    if stripped.replace(" ", "") in {"목차", "차례"}:
        return True

    # 페이지 번호만 있는 줄
    if re.fullmatch(r"\d+", stripped):
        return True

    # 목차의 dot leader
    if re.search(r"(\.{3,}|…{2,}|·{3,})", stripped):
        return True

    return False


def make_article_no(article_num: str, sub_num: str | None) -> str:
    """
    제1조, 제1조의2 형태 생성.
    """
    if sub_num:
        return f"제{article_num}조의{sub_num}"
    return f"제{article_num}조"


def looks_like_article_heading(line: str, match: re.Match) -> bool:
    """
    단순 참조문인 '제10조에 따라...'를 조문 제목으로 오인하지 않도록 필터링.
    """
    title = (match.group(3) or "").strip()
    rest = (match.group(4) or "").strip()

    # 제1조(목적) 형태면 거의 확실한 조문 제목
    if title:
        return True

    # 제10조 삭제
    if rest.startswith("삭제"):
        return True

    # 제목 괄호가 없는 규정도 있으므로 짧은 제목형 문장은 허용
    # 단, '제10조에 따라', '제10조의 규정' 같은 참조문은 제외
    if len(rest) <= 60 and not re.match(r"^(에|에서|의|을|를|은|는|와|과|및|또는|부터|까지)", rest):
        return True

    return False


def make_org_metadata(no: str, title: str, unit: str) -> dict:
    """
    장/절 정보를 dict로 정리.
    """
    title = re.sub(r"\s+", " ", title).strip()
    raw = f"제{no}{unit}" + (f" {title}" if title else "")

    return {
        "no": f"제{no}{unit}",
        "title": title if title else None,
        "raw": raw
    }

# =========================
# 2단 구조 함수
# =========================

def extract_text_two_columns(page, column_gap_ratio: float = 0.5) -> str:
    """
    2단 PDF 페이지에서 텍스트 블록 좌표를 기준으로
    왼쪽 단 → 오른쪽 단 순서로 텍스트를 추출한다.

    PyMuPDF block 구조:
    block = (x0, y0, x1, y1, text, block_no, block_type)
    """

    page_width = page.rect.width
    middle_x = page_width * column_gap_ratio

    blocks = page.get_text("blocks")

    left_blocks = []
    right_blocks = []
    full_width_blocks = []

    for block in blocks:
        x0, y0, x1, y1, text, block_no, block_type = block

        text = text.strip()
        if not text:
            continue

        # block_type 0은 일반 텍스트
        if block_type != 0:
            continue

        block_center_x = (x0 + x1) / 2

        # 장 제목, 절 제목처럼 페이지 전체 폭에 걸친 블록일 수 있음
        block_width = x1 - x0
        is_full_width = block_width > page_width * 0.65

        if is_full_width:
            full_width_blocks.append(block)
        elif block_center_x < middle_x:
            left_blocks.append(block)
        else:
            right_blocks.append(block)

    # 전체 폭 블록은 보통 상단 제목/장/절 제목일 가능성이 있음
    full_width_blocks.sort(key=lambda b: (b[1], b[0]))

    # 각 단 내부는 위에서 아래, 같은 높이면 왼쪽에서 오른쪽
    left_blocks.sort(key=lambda b: (b[1], b[0]))
    right_blocks.sort(key=lambda b: (b[1], b[0]))

    ordered_blocks = full_width_blocks + left_blocks + right_blocks

    page_text = "\n".join(block[4].strip() for block in ordered_blocks)

    return page_text


# =========================
# 핵심 파싱 함수
# =========================

def parse_regulation_pdf(
    pdf_path: str,
    start_page: int = 1,
    end_page: int | None = None,
    include_supplementary: bool = False
) -> list[dict]:
    """
    PDF 규정집을 제n조 기준으로 청킹한다.

    metadata:
    - file_title
    - chapter
    - section
    """

    pdf_path = Path(pdf_path)

    chunks = []
    current_chunk = None
    current_chapter = None
    current_section = None

    with fitz.open(pdf_path) as doc:
        pdf_meta_title = (doc.metadata or {}).get("title", "")
        file_title = pdf_meta_title.strip() or pdf_path.stem

        total_pages = len(doc)
        if end_page is None:
            end_page = total_pages

        def flush_current_chunk():
            """
            현재 조문 chunk를 chunks에 저장하고 초기화.
            """
            nonlocal current_chunk

            if current_chunk is None:
                return

            content = "\n".join(current_chunk.pop("_content_lines")).strip()

            if content:
                current_chunk["content"] = content
                chunks.append(current_chunk)

            current_chunk = None

        for page_no in range(start_page, end_page + 1):
            page = doc[page_no - 1]
            # 2단 PDF용 추출
            text = extract_text_two_columns(page)

            # 전처리
            text = normalize_page_text(text)

            for raw_line in text.splitlines():
                line = raw_line.strip()

                if is_noise_line(line):
                    continue

                # -------------------------
                # 장 감지: 제1장 총칙
                # -------------------------
                chapter_match = CHAPTER_RE.match(line)

                if chapter_match and len(line) <= 80:
                    flush_current_chunk()

                    current_chapter = make_org_metadata(
                        no=chapter_match.group(1),
                        title=chapter_match.group(2),
                        unit="장"
                    )

                    # 장이 바뀌면 절은 초기화
                    current_section = None
                    continue

                # -------------------------
                # 절 감지: 제1절 통칙
                # -------------------------
                section_match = SECTION_RE.match(line)

                if section_match and len(line) <= 80:
                    flush_current_chunk()

                    current_section = make_org_metadata(
                        no=section_match.group(1),
                        title=section_match.group(2),
                        unit="절"
                    )
                    continue

                # -------------------------
                # 부칙 감지
                # -------------------------
                supplementary_match = SUPPLEMENTARY_RE.match(line)

                if supplementary_match:
                    flush_current_chunk()

                    if include_supplementary:
                        current_chunk = {
                            "chunk_type": "supplementary",
                            "article_no": "부칙",
                            "article_title": None,
                            "metadata": {
                                "file_title": file_title,
                                "chapter": None,
                                "section": None
                            },
                            "start_page": page_no,
                            "end_page": page_no,
                            "_content_lines": [line]
                        }
                    else:
                        current_chunk = None

                    continue

                # -------------------------
                # 조문 감지: 제1조(목적)
                # -------------------------
                article_match = ARTICLE_RE.match(line)

                if article_match and looks_like_article_heading(line, article_match):
                    flush_current_chunk()

                    article_no = make_article_no(
                        article_match.group(1),
                        article_match.group(2)
                    )

                    article_title = (article_match.group(3) or "").strip()
                    article_title = article_title if article_title else None

                    current_chunk = {
                        "chunk_type": "article",
                        "article_no": article_no,
                        "article_title": article_title,
                        "metadata": {
                            "file_title": file_title,
                            "chapter": current_chapter["raw"] if current_chapter else None,
                            "section": current_section["raw"] if current_section else None
                        },
                        "start_page": page_no,
                        "end_page": page_no,
                        "_content_lines": [line]
                    }

                    continue

                # -------------------------
                # 일반 본문 라인
                # -------------------------
                if current_chunk is not None:
                    current_chunk["_content_lines"].append(line)
                    current_chunk["end_page"] = page_no

        flush_current_chunk()

    return chunks

In [13]:
chunks = parse_regulation_pdf(
    pdf_path=pdf_path,
    start_page=START_PAGE,
    end_page=END_PAGE,
    include_supplementary=INCLUDE_SUPPLEMENTARY
)

print("생성된 chunk 개수:", len(chunks))

output_path = f"{Path(pdf_path).stem}_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("저장 완료:", output_path)

colab_files.download(output_path)

생성된 chunk 개수: 18
저장 완료: 2110 사규관리규정(제17차 개정 20201120) (1)_chunks.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
preview = pd.DataFrame([
    {
        "article_no": c.get("article_no"),
        "article_title": c.get("article_title"),
        "file_title": c["metadata"].get("file_title"),
        "chapter": c["metadata"].get("chapter"),
        "section": c["metadata"].get("section"),
        "start_page": c.get("start_page"),
        "end_page": c.get("end_page"),
        "content_preview": c.get("content", "")[:120]
    }
    for c in chunks
])

preview.head(20)

,article_no,article_title,file_title,chapter,section,start_page,end_page,content_preview
0,제1조,목적,2110 사규관리규정(제17차 개정 20201120) (1),제1장 총 칙,None,1,1,제1조 (목적) 이 규정은 사규의 제정ㆍ개폐 및 공\n포에 따르는 필요한 사항을 규...
1,제2조,용어의 정의,2110 사규관리규정(제17차 개정 20201120) (1),제1장 총 칙,None,1,1,제2조 (용어의 정의) 이 규정에서 사용하는 용어의\n정의는 다음과 같다.\n1. ...
2,제3조,사규화,2110 사규관리규정(제17차 개정 20201120) (1),제1장 총 칙,None,1,1,제3조 (사규화) ① 회사의 조직과 업무운영에 준거\n할 제규범과 기준은 이 규정이...
3,제4조,None,2110 사규관리규정(제17차 개정 20201120) (1),제2장 사규체제,None,1,1,제4조 <삭제 2008.4.10>
4,제5조,명칭 및 번호,2110 사규관리규정(제17차 개정 20201120) (1),제2장 사규체제,None,1,1,"제5조 (명칭 및 번호) 사규에는 그 내용을 적절,\n사규관리규정\n사규관리규정\n..."
5,제6조,작성형식,2110 사규관리규정(제17차 개정 20201120) (1),제2장 사규체제,None,1,1,제6조 (작성형식) 사규의 작성형식은 다음 각호와\n같다.\n1. 사규는 목차ㆍ총칙...
6,제7조,제정 및 개폐절차,2110 사규관리규정(제17차 개정 20201120) (1),제3장 사규의 제정과 개폐,None,1,2,제7조 (제정 및 개폐절차) ① 사규를 제정ㆍ개폐\n하고자 할 때에는 다음 각호의 ...
7,제7조의2,중요 또는 경미한 내용의 판단기준,2110 사규관리규정(제17차 개정 20201120) (1),제3장 사규의 제정과 개폐,None,2,2,제7조의2 (중요 또는 경미한 내용의 판단기준)\n사규의 제정 및 개폐에 있어 직무...
8,제7조의3,사규 제·개정안 사전예고,2110 사규관리규정(제17차 개정 20201120) (1),제3장 사규의 제정과 개폐,None,2,2,제7조의3 (사규 제·개정안 사전예고) ① 업무주관\n부서는 제․개정하고자 하는 사...
9,제7조의4,사전예고에 대한 의견제출 및 처리,2110 사규관리규정(제17차 개정 20201120) (1),제3장 사규의 제정과 개폐,None,2,2,제7조의4 (사전예고에 대한 의견제출 및 처리)\n① 사전예고된 사규안에 대하여 이...
